In [ ]:
from IPython.display import HTML
display(HTML("<style>.rendered_html { font-size: 1.3em; } .code_cell .input_area { font-size: 1.1em; }</style>"))

# 7.6 Get Your Survey Data Machine Learning Ready
- ColumnTransformer for structured features
- PCA to compress TF-IDF text down to a manageable size
- Fusing structured + text into one master matrix

## Setup

In [ ]:
import random
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA

pd.options.display.max_columns = None
np.random.seed(42)
random.seed(42)

df_training = pd.read_csv('../data/training.csv')
df_testing = pd.read_csv('../data/testing.csv')
print("Training size:", len(df_training))
print("Test size:", len(df_testing))

## Preprocess Structured Student Data
MinMaxScaler for academic ratios (preserves 0–4 bounds), StandardScaler for unit counts, OneHotEncoder for demographics.

In [ ]:
minmax_cols = ['HS_GPA', 'GPA_1', 'GPA_2', 'DFW_RATE_1', 'DFW_RATE_2']
standard_cols = ['UNITS_ATTEMPTED_1', 'UNITS_ATTEMPTED_2']
categorical_cols = ['GENDER', 'RACE_ETHNICITY', 'FIRST_GEN_STATUS']

df_train = df_training.dropna(subset=minmax_cols + standard_cols + categorical_cols).copy()
df_test = df_testing.dropna(subset=minmax_cols + standard_cols + categorical_cols).copy()

preprocessor = ColumnTransformer(
    transformers=[
        ('minmax', MinMaxScaler(), minmax_cols),
        ('standard', StandardScaler(), standard_cols),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
    ],
    remainder='drop'
)

X_structured_train = preprocessor.fit_transform(df_train)
df_structured_train = pd.DataFrame(X_structured_train, index=df_train.index)

X_structured_test = preprocessor.transform(df_test)
df_structured_test = pd.DataFrame(X_structured_test, index=df_test.index)

print("Structured feature matrix:", df_structured_train.shape)

## Reduce TF-IDF Text Features with PCA
80% cumulative explained variance is a common IR threshold. PCA is fit on train only, then `.transform()` on test — never `.fit_transform()` on test, to avoid leakage.

In [ ]:
ML_Survey_Data_Num = pd.read_csv('../data/ML_Survey_Data_Num.csv')
ML_Survey_Data22_Num = pd.read_csv('../data/ML_Survey_Data22_Num.csv')

tfidf_matrix_train = ML_Survey_Data_Num.iloc[:, 11:]
tfidf_matrix_test = ML_Survey_Data22_Num.iloc[:, 11:]
print("TF-IDF matrix:", tfidf_matrix_train.shape)

In [ ]:
pca_full = PCA(random_state=42)
pca_full.fit(tfidf_matrix_train)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
threshold = 0.80
n_components = int(np.searchsorted(cumvar, threshold)) + 1
print(f"Using {n_components} PCA components to capture {threshold*100:.0f}% of text variance")

pca = PCA(n_components=n_components, random_state=42)
X_text_pca_train = pca.fit_transform(tfidf_matrix_train)
X_text_pca_test = pca.transform(tfidf_matrix_test)  # transform only — never fit on test

df_text_pca_train = pd.DataFrame(X_text_pca_train, index=df_train.index,
                                  columns=[f'TEXT_PC{i+1}' for i in range(n_components)])
df_text_pca_test = pd.DataFrame(X_text_pca_test, index=df_test.index,
                                 columns=[f'TEXT_PC{i+1}' for i in range(n_components)])
print("Text PCA matrix:", df_text_pca_train.shape)

## Feature Fusion: Creating the Master Matrix

In [ ]:
df_all_train = pd.concat([df_structured_train, df_text_pca_train], axis=1)
df_all_test = pd.concat([df_structured_test, df_text_pca_test], axis=1)
df_all_train.columns = df_all_train.columns.astype(str)
df_all_test.columns = df_all_test.columns.astype(str)

transformed_structured_feature_names = preprocessor.get_feature_names_out()
cleaned_structured_cols = [c.split('__', 1)[-1] for c in transformed_structured_feature_names]
pca_cols = df_text_pca_train.columns.tolist()

df_all_train.columns = cleaned_structured_cols + pca_cols
df_all_test.columns = cleaned_structured_cols + pca_cols

df_all_train['SEM_3_STATUS'] = df_training.loc[df_train.index, 'SEM_3_STATUS']
df_all_test['SEM_3_STATUS'] = df_testing.loc[df_test.index, 'SEM_3_STATUS']

print("Combined feature matrix:", df_all_train.shape)
df_all_train.head(3)

## Export

In [ ]:
df_all_train.to_csv('../data/ML_SURVEY_MASTER_TRAIN.csv', index=False)
df_all_test.to_csv('../data/ML_SURVEY_MASTER_TEST.csv', index=False)
print("Saved ML_SURVEY_MASTER_TRAIN.csv and ML_SURVEY_MASTER_TEST.csv to ../data/")

## Summary
- Structured features scaled via ColumnTransformer, text features compressed via PCA (fit-on-train-only).
- Fused into one master matrix — this is the exact input Module 8 uses for supervised and unsupervised modeling.

**Next:** Module 8 builds on this master matrix directly.